For task 3, we will build and train a neural network model on the data. We will experiment with our hyperparameters until we find a model that performs as well as the principal component logistic regression from task 1.

In specific, we are using the new cleaned dataset that has paragraph and header content, not the original dataset that only has header content.

We will attempt to surpass these baseline metrics from the principal component logistic regression trained on the paragraph and header data:

Accuracy: 0.8271028037383178

AUC: 0.8934490664919751

Sensitivity: 0.8403755868544601

Specificity: 0.813953488372093

For reference, here is the confusion matrix for that model as well:

|   | Predicted Irrelevant | Predicted Relevant |
| --- | --- | --- |
| Actual Irrelevant  | 175 | 40 |
| Actual Relevant  | 34 | 179 |

First we import some libraries:

In [16]:
import pyreadr
import pandas as pd
import re
import xml
from bs4 import BeautifulSoup
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, classification_report, confusion_matrix
import numpy as np
from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from time import perf_counter
from sklearn.neural_network import MLPClassifier

In [3]:
# Import dataset
raw_claims_df = pyreadr.read_r("../../data/claims-raw.RData")["claims_raw"]
raw_claims_df.shape

(2165, 9)

In [4]:
# Code from task 1 to reproduce same cleaned dataset
def parse_fn(html, selector = None):
    soup = BeautifulSoup(html, features = "lxml")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements

        if selector is None:
            raw = ' '.join(spoon.get_text(' '))
        else:
            raw = ' '.join(spoon.get_text(' ') for spoon in soup.select(selector))
        
        # remove urls and emails
            # \S@\S+ means non-whitespace characters followed by @ then followed by 1 or more non-whitespace characters
            # pattern = r'https?://\S+|www\.\S+' which means http with optional s followed by :// and then 1 or more non-whitespace characters
            # |www\.\S+ means or www. then 1 or more non-whitespace characters
        text = re.sub(r'\S+@\S+|https?://\S+|www\.\S+', "", raw)

        # remove apostraphes

        text = re.sub("'", "",text)

        # remove non-alphabetic characters and nbsp
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # add space between camel case words

        text = text.lower()

        text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces into one and strip leading/trailing whitespace
        # remove punctuation and newlines
        
        return text 
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None


def parse_data(df, selector = None):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() # filter for rows that contain html tags
    df['text_clean'] = df['text_tmp'].apply(parse_fn, selector = selector) # apply parse_fn and create a new column
    return df

In [5]:
# Apply function to store paragraph and header data
clean_claims_df = parse_data(raw_claims_df, selector = 'h1, h2, h3, h4, h5, h6, p')

C:\Users\posit\AppData\Local\Temp\ipykernel_3836\3430768308.py:3: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, features = "lxml")


In [6]:
# Match task 1 split
train_df, test_df = train_test_split(
    clean_claims_df,
    test_size=0.2,
    random_state=101422
)

# Use only training pages for architecture experiments.
fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["bclass"]
)

# Output length of training, fitting, validation, and test datasets
print("Full training:", len(train_df))
print("Fitting:", len(fit_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Full training: 1712
Fitting: 1369
Validation: 343
Test: 428


In [10]:
# Lemmatizer for nn
lemmatizer = WordNetLemmatizer()
lemmatizer.lemmatize("claims")

# stop words
stop_words = {word.replace("'", "") for word in ENGLISH_STOP_WORDS}

# Tokenizing function
def tokenize_words(text):
    return [
        lemmatizer.lemmatize(token)
        for token in text.split()
        if token not in stop_words
    ]

# convert to tf-idf matrix
nn_vectorizer = TfidfVectorizer(
    tokenizer=tokenize_words,
    lowercase=False,
    token_pattern=None,
    min_df=2,
    max_features=10000,
    dtype=np.float32
)

# Learn preprocessing from the fitting subset only.
nn_x_fit = nn_vectorizer.fit_transform(fit_df["text_clean"])
nn_x_val = nn_vectorizer.transform(val_df["text_clean"])

nn_y_fit = fit_df["bclass"].eq(
    "Relevant claim content"
).astype(int).to_numpy()

nn_y_val = val_df["bclass"].eq(
    "Relevant claim content"
).astype(int).to_numpy()

print("Fitting features:", nn_x_fit.shape)
print("Validation features:", nn_x_val.shape)

Fitting features: (1369, 10000)
Validation features: (343, 10000)


First we will try this neural network with some set of set hyperparameters and see how it stacks up against the PC Logistic Regression. For this neural network we went with the following design choices:

* One hidden layer with 32 neurons
* Activation function: ReLU
* Ridge regression penalty: 0.001
* Learning rate: 0.001
* Maximum number of training epochs: 100

For each epoch we will record the iteration number and loss function.

In [12]:
nn_first = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=False,
    random_state=42,
    verbose=True
)

start = perf_counter()
nn_first.fit(nn_x_fit, nn_y_fit)

print(f"Training time: {perf_counter() - start:.1f} seconds")
print("Epochs completed:", nn_first.n_iter_)

Iteration 1, loss = 0.66961399
Iteration 2, loss = 0.57450057
Iteration 3, loss = 0.46488783
Iteration 4, loss = 0.37764576
Iteration 5, loss = 0.31439997
Iteration 6, loss = 0.26919029
Iteration 7, loss = 0.23481047
Iteration 8, loss = 0.20956062
Iteration 9, loss = 0.18936812
Iteration 10, loss = 0.17222647
Iteration 11, loss = 0.16086062
Iteration 12, loss = 0.14864286
Iteration 13, loss = 0.13886864
Iteration 14, loss = 0.13035884
Iteration 15, loss = 0.12384660
Iteration 16, loss = 0.11823267
Iteration 17, loss = 0.11299718
Iteration 18, loss = 0.10865383
Iteration 19, loss = 0.10457762
Iteration 20, loss = 0.10131672
Iteration 21, loss = 0.09810308
Iteration 22, loss = 0.09542098
Iteration 23, loss = 0.09198411
Iteration 24, loss = 0.09052459
Iteration 25, loss = 0.08719777
Iteration 26, loss = 0.08604488
Iteration 27, loss = 0.08426042
Iteration 28, loss = 0.08186983
Iteration 29, loss = 0.08061983
Iteration 30, loss = 0.07960559
Iteration 31, loss = 0.07797819
Iteration 32, los

C:\Users\posit\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [13]:
# Function to calculate model metrics
def classification_metrics(model, x, y, dataset):
    positive_column = np.flatnonzero(model.classes_ == 1)[0]
    probabilities = model.predict_proba(x)[:, positive_column]
    predictions = (probabilities > 0.5).astype(int)

    return {
        "dataset": dataset,
        "accuracy": accuracy_score(y, predictions),
        "roc_auc": roc_auc_score(y, probabilities),
        "sensitivity": recall_score(y, predictions, zero_division=0),
        "specificity": recall_score(
            y, predictions, pos_label=0, zero_division=0
        )
    }

first_nn_results = pd.DataFrame([
    classification_metrics(
        nn_first, nn_x_fit, nn_y_fit, "Fitting"
    ),
    classification_metrics(
        nn_first, nn_x_val, nn_y_val, "Validation"
    )
])

first_nn_results

,dataset,accuracy,roc_auc,sensitivity,specificity
0,Fitting,0.985391,0.999385,0.991758,0.978159
1,Validation,0.807580,0.860929,0.846995,0.762500


As we can see, this model does not do as well as the PC Logistic Regression model on the testing set. It has lower accuracy (0.80758 to 0.8271), AUC ROC (0.86 to 0.89), and specificity (0.7625 to 0.814). The only statistic it improves upon is sensitivity (0.847 to 0.840), though the gain is very marginal.

What sticks out immediately is the fact that there is a huge drpo in all the statistics when going from the fitting set to the validation set. This suggests that the model is overfitting.

For this, we can try a few things to prevent overfitting:

* Increasing the ridge regression penalty, $\alpha$
* Reducing the hidden layer width from 32 to 16
* Loewring the number of epochs the neural network uses to train.

In [14]:
# Tests configurations: stronger ridge penalthy, lower hidden layer width, original settings
experiments = [
    {"name": "Original settings", "width": 32, "alpha": 0.001},
    {"name": "Stronger L2", "width": 32, "alpha": 0.05},
    {"name": "Smaller layer", "width": 16, "alpha": 0.001},
]

# Test configuration: lowering epochs
checkpoints = {5, 10, 20, 40}
rows = []

# Run each configuration and see which one does the best on the validation set
for config in experiments:
    model = MLPClassifier(
        hidden_layer_sizes=(config["width"],),
        activation="relu",
        solver="adam",
        alpha=config["alpha"],
        batch_size=32,
        learning_rate_init=0.001,
        early_stopping=False,
        random_state=np.random.RandomState(42),
    )

    for epoch in range(1, max(checkpoints) + 1):
        # partial_fit runs one epoch and continues from the previous weights.
        model.partial_fit(
            nn_x_fit, nn_y_fit, classes=np.array([0, 1])
        )

        if epoch in checkpoints:
            val = classification_metrics(
                model, nn_x_val, nn_y_val, "Validation"
            )
            fit_accuracy = model.score(nn_x_fit, nn_y_fit)

            rows.append({
                "experiment": config["name"],
                "width": config["width"],
                "alpha": config["alpha"],
                "epochs": epoch,
                "fit_accuracy": fit_accuracy,
                "val_accuracy": val["accuracy"],
                "val_auc": val["roc_auc"],
                "val_sensitivity": val["sensitivity"],
                "val_specificity": val["specificity"],
            })

experiment_results = pd.DataFrame(rows)

display(experiment_results.round(4))

print("Top configurations by validation accuracy:")
display(
    experiment_results
    .sort_values(
        ["val_accuracy", "val_auc", "epochs"],
        ascending=[False, False, True],
    )
    .head(5)
    .round(4)
)

,experiment,width,alpha,epochs,fit_accuracy,val_accuracy,val_auc,val_sensitivity,val_specificity
0,Original settings,32,0.001,5,0.9372,0.8251,0.8864,0.8470,0.8000
1,Original settings,32,0.001,10,0.9598,0.8163,0.8816,0.8251,0.8062
2,Original settings,32,0.001,20,0.9737,0.8047,0.8733,0.8251,0.7812
3,Original settings,32,0.001,40,0.9832,0.8047,0.8620,0.8197,0.7875
4,Stronger L2,32,0.050,5,0.9189,0.8338,0.8869,0.8689,0.7938
5,Stronger L2,32,0.050,10,0.9416,0.8222,0.8849,0.8361,0.8062
6,Stronger L2,32,0.050,20,0.9576,0.8163,0.8811,0.8415,0.7875
7,Stronger L2,32,0.050,40,0.9708,0.8222,0.8782,0.8306,0.8125
8,Smaller layer,16,0.001,5,0.9291,0.8192,0.8841,0.8361,0.8000
9,Smaller layer,16,0.001,10,0.9496,0.8192,0.8824,0.8197,0.8188


Top configurations by validation accuracy:


,experiment,width,alpha,epochs,fit_accuracy,val_accuracy,val_auc,val_sensitivity,val_specificity
4,Stronger L2,32,0.050,5,0.9189,0.8338,0.8869,0.8689,0.7938
0,Original settings,32,0.001,5,0.9372,0.8251,0.8864,0.8470,0.8000
5,Stronger L2,32,0.050,10,0.9416,0.8222,0.8849,0.8361,0.8062
7,Stronger L2,32,0.050,40,0.9708,0.8222,0.8782,0.8306,0.8125
8,Smaller layer,16,0.001,5,0.9291,0.8192,0.8841,0.8361,0.8000


From the above table, it looks like our best models from these configuration options improve upon the validation accuracy which suggests some of these hyperparameters reduced overfitting. The best performing model on the validation set has the following configuration:

* Hidden layer width: 32
* Ridge Penalty: $\alpha = 0.05$
* Epochs: 5

We will go with this model and see if it can perform better than the PC logistic regression.

In [15]:
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

# Learn a fresh vocabulary and IDF weights from all training pages.
final_vectorizer = clone(nn_vectorizer)
final_x_train = final_vectorizer.fit_transform(train_df["text_clean"])
final_x_test = final_vectorizer.transform(test_df["text_clean"])

positive_label = "Relevant claim content"
final_y_train = train_df["bclass"].eq(positive_label).astype(int).to_numpy()
final_y_test = test_df["bclass"].eq(positive_label).astype(int).to_numpy()

final_nn = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="relu",
    solver="adam",
    alpha=0.05,
    batch_size=32,
    learning_rate_init=0.001,
    early_stopping=False,
    random_state=np.random.RandomState(42),
)

for _ in range(5):
    final_nn.partial_fit(
        final_x_train, final_y_train, classes=np.array([0, 1])
    )

nn_test_metrics = classification_metrics(
    final_nn, final_x_test, final_y_test, "Test"
)

comparison = pd.DataFrame([
    {
        "model": "Task 1 SVD logistic regression",
        "accuracy": 354 / 428,
        "roc_auc": 0.8934490664919751,
        "sensitivity": 179 / 213,
        "specificity": 175 / 215,
    },
    {
        "model": "Task 3 neural network",
        **{
            metric: nn_test_metrics[metric]
            for metric in ["accuracy", "roc_auc", "sensitivity", "specificity"]
        },
    },
])

display(comparison.round(4))

accuracy_change = 100 * (nn_test_metrics["accuracy"] - 354 / 428)
print(f"Accuracy change: {accuracy_change:+.2f} percentage points")

test_prob = final_nn.predict_proba(final_x_test)[:, 1]
test_pred = (test_prob > 0.5).astype(int)
print("Neural network confusion matrix:")
print(confusion_matrix(final_y_test, test_pred, labels=[0, 1]))

,model,accuracy,roc_auc,sensitivity,specificity
0,Task 1 SVD logistic regression,0.8271,0.8934,0.8404,0.8140
1,Task 3 neural network,0.8364,0.9010,0.8451,0.8279


Accuracy change: +0.93 percentage points
Neural network confusion matrix:
[[178  37]
 [ 33 180]]


This neural network performed slightly better in terms of accuracy, ROC AUC, sensitivity, and specificity than the PC logistic regression. Below is the confusion matrix:

|   | Predicted Irrelevant | Predicted Relevant |
| --- | --- | --- |
| Actual Irrelevant  | 178 | 37 |
| Actual Relevant | 33 | 180 |